In [224]:
from ollama import Client
import os
from pydantic import BaseModel, ValidationError
import re
import json
import pandas as pd
import numpy as np

import ast
from typing import List, Dict, Tuple, Optional
from collections import namedtuple

import csv  # 添加这一行

In [159]:
# 定义ImportEntry结构
ImportEntry = namedtuple('ImportEntry', ['module', 'asname', 'imported_names'])

In [160]:
class Function:
    func_id: int = 0
    func_name: str
    func_desc: str = ""
    func_file: str = ""
    func_flow: str = ""
    func_notf: str = ""
    func_code: str
    func_fullName: str 
    func_txt_vector: List[float] = []

    def __init__(self, func_name, func_desc, func_code, func_fullName):
        self.func_name = func_name
        self.func_desc = func_desc
        self.func_code = func_code
        self.func_fullName = func_fullName

In [161]:
class File:
    file_id: int = 0
    file_name: str
    file_path: str
    file_code: str = ""
    file_desc: str = ""
    func_list: List[Function] = []
    file_txt_vector: List[float] = []
    file_discode: str = ""
    import_list: List[ImportEntry] = []
    module_name: str = ""

    def __init__(self, file_name, file_path, file_code, file_desc):
        self.file_name = file_name
        self.file_path = file_path
        self.file_code = file_code  
        self.file_desc = file_desc
        self.func_list = []
        self.import_list = []

In [193]:
class CodeAnalyzer:
    def __init__(self, base_dir):
        self.base_dir = base_dir
        self.files: List[File] = []
        self.functions: List[Function] = []
        self.module_to_file_map: Dict[str, File] = {}
        self.func_name_to_obj: Dict[str, List[Function]] = {}
        self.func_adj_matrix: Optional[np.ndarray] = None
        self.file_adj_matrix: Optional[np.ndarray] = None
        self.current_file: Optional[File] = None

    def analyze_project(self):
        """分析整个项目"""
        # 第一步：收集所有Python文件
        self._collect_python_files()
        
        # 第二步：建立模块名到文件的映射
        self._build_module_mapping()
        
        # 第三步：分配ID并创建函数映射
        self._assign_ids_and_build_mappings()
        
        # 第四步：构建文件导入关系矩阵
        self._build_file_adj_matrix()
        
        # 第五步：构建函数调用关系矩阵
        self._build_func_adj_matrix()
        
        return self.files, self.functions, self.file_adj_matrix, self.func_adj_matrix

    def _collect_python_files(self):
        """收集所有Python文件并解析基本结构"""
        for root, _, files in os.walk(self.base_dir):
            for file in files:
                if file.endswith('.py'):
                    file_path = os.path.join(root, file)
                    try:
                        with open(file_path, 'r', encoding='utf-8') as f:
                            code = f.read()
                        self._parse_python_file(file_path, code)
                    except Exception as e:
                        print(f"Error parsing {file_path}: {str(e)}")

    def _parse_python_file(self, file_path: str, code: str):
        """解析单个Python文件"""
        try:
            tree = ast.parse(code)
        except SyntaxError:
            return

        # 提取文件描述（模块级docstring）
        file_desc = ""
        if tree.body and isinstance(tree.body[0], ast.Expr) and isinstance(tree.body[0].value, ast.Str):
            file_desc = tree.body[0].value.s

        file_name = os.path.basename(file_path)
        file_obj = File(file_name, file_path, code, file_desc)
        self.current_file = file_obj

        # 先计算并设置模块名（关键修复）
        rel_path = os.path.relpath(file_path, self.base_dir)
        module_name = os.path.splitext(rel_path)[0].replace(os.sep, '.')
        if file_name == '__init__.py':
            module_name = module_name.replace('.__init__', '')
        file_obj.module_name = module_name

        # 解析导入语句
        for node in ast.walk(tree):
            if isinstance(node, (ast.Import, ast.ImportFrom)):
                self._process_import(node)
        
        # 解析函数定义
        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):
                self._process_function(node, code)
        
        self.files.append(file_obj)
        self.current_file = None

    # 改进导入处理函数
    def _process_import(self, node):
        """处理导入语句，支持相对导入和别名"""
        # 处理标准import
        if isinstance(node, ast.Import):
            for alias in node.names:
                module_name = alias.name
                asname = alias.asname or module_name.split('.')[0]
            
                # 存储导入信息
                self.current_file.import_list.append(
                ImportEntry(module=module_name, asname=asname, imported_names=None)
            )
    
        # 处理from...import
        elif isinstance(node, ast.ImportFrom):
            module_name = node.module or ''
            level = node.level  # 相对导入级别
        
            # 处理相对导入
            if level > 0:
                current_module = self.current_file.module_name
                parts = current_module.split('.')

                # 计算基础模块路径
                if level <= len(parts):
                    base_module = '.'.join(parts[:-level])
                    # 合并基础模块和导入模块
                    if module_name:
                        module_name = f"{base_module}.{module_name}"
                    else:
                        module_name = base_module
                print(f"Relative import: level={level}, base={base_module}, final={module_name}")

             # 处理通配符导入(*)
            imported_names = []
            for alias in node.names:
                if alias.name == '*':
                    # 特殊处理通配符导入
                    self.current_file.import_list.append(
                        ImportEntry(module=module_name, asname=None, imported_names="*")
                    )
                    print(f"Wildcard import: from {module_name} import *")
                else:
                    imported_name = alias.name
                    asname = alias.asname or imported_name
                    imported_names.append((imported_name, asname))
            
            # 处理普通导入
            if imported_names:
                self.current_file.import_list.append(
                    ImportEntry(module=module_name, asname=None, imported_names=imported_names)
                )
                print(f"From import: from {module_name} import {imported_names}")

    def _process_function(self, node, code: str):
        """处理函数定义"""
        func_name = node.name
        
        # 获取函数描述（docstring）
        func_desc = ast.get_docstring(node) or ""
        
        # 获取函数源代码
        func_code = ast.get_source_segment(code, node) or ""
        
        # 生成函数全名
        func_fullName = f"{self.current_file.module_name}.{func_name}" if self.current_file.module_name else func_name
        
        # 创建函数对象
        func_obj = Function(func_name, func_desc, func_code, func_fullName)
        func_obj.func_file = self.current_file.file_path
        
        # 添加到文件对象和全局列表
        self.current_file.func_list.append(func_obj)
        self.functions.append(func_obj)

    def _build_module_mapping(self):
        """建立模块名到文件的映射（修复映射冲突）"""
        self.module_to_file_map = {}
        
        # 先处理所有__init__.py文件
        init_files = [f for f in self.files if f.file_name == '__init__.py']
        for file in init_files:
            # 对于__init__.py，模块名就是包名
            self.module_to_file_map[file.module_name] = file
            print(f"Mapped package: {file.module_name} -> {file.file_name}")
        
        # 再处理普通文件
        for file in self.files:
            if file not in init_files:  # 跳过已处理的__init__.py
                # 直接映射文件模块名
                self.module_to_file_map[file.module_name] = file
                print(f"Mapped module: {file.module_name} -> {file.file_name}")

    def _assign_ids_and_build_mappings(self):
        """分配ID并创建函数名到对象的映射"""
        # 为文件分配ID
        for idx, file in enumerate(self.files):
            file.file_id = idx
        
        # 为函数分配ID并创建映射
        self.func_name_to_obj = {}
        for idx, func in enumerate(self.functions):
            func.func_id = idx
            if func.func_name not in self.func_name_to_obj:
                self.func_name_to_obj[func.func_name] = []
            self.func_name_to_obj[func.func_name].append(func)


    # 改进文件关系矩阵构建
    def _build_file_adj_matrix(self):
        """构建文件导入关系矩阵（增强匹配逻辑）"""
        n = len(self.files)
        self.file_adj_matrix = np.zeros((n, n), dtype=int)
        
        # 创建文件路径到ID的映射
        file_path_to_id = {f.file_path: f.file_id for f in self.files}
        
        for i, file in enumerate(self.files):
            for imp in file.import_list:
                module_name = imp.module
                if not module_name:
                    continue
                
                # 尝试匹配的模块名变体
                candidates = [
                    module_name,
                    module_name.replace('_', '-'),  # 处理命名风格差异
                    module_name.split('.')[-1]       # 尝试最后部分
                ]
                
                target_file = None
                for candidate in candidates:
                    if candidate in self.module_to_file_map:
                        target_file = self.module_to_file_map[candidate]
                        break
                
                # 如果未找到，尝试更宽松的匹配
                if not target_file:
                    for mapped_module, mapped_file in self.module_to_file_map.items():
                        if mapped_module.endswith('.' + module_name) or module_name.endswith('.' + mapped_module):
                            target_file = mapped_file
                            break
                
                if target_file:
                    j = target_file.file_id
                    if i != j:  # 排除自引用
                        self.file_adj_matrix[i][j] = 1
                        print(f"Import found: {file.file_name} -> {target_file.file_name} ({module_name})")
                    else:
                        print(f"Self import ignored: {file.file_name} imports itself")
                else:
                    print(f"Import unresolved: {file.file_name} imports {module_name}")

    def _build_func_adj_matrix(self):
        """构建函数调用关系矩阵"""
        n = len(self.functions)
        self.func_adj_matrix = np.zeros((n, n), dtype=int)
        
        # 为每个文件创建函数名到对象的映射
        file_func_map = {}
        for file in self.files:
            file_func_map[file.file_path] = {f.func_name: f for f in file.func_list}
        
        for caller_idx, caller_func in enumerate(self.functions):
            caller_file = self._find_file_by_path(caller_func.func_file)
            if not caller_file:
                continue
            
            # 创建当前作用域的符号表
            symbol_table = self._build_symbol_table(caller_file)
            
            # 解析函数体
            try:
                tree = ast.parse(caller_func.func_code)
                for node in ast.walk(tree):
                    if isinstance(node, ast.Call):
                        callee_func = self._resolve_function_call(node, symbol_table, file_func_map)
                        if callee_func:
                            callee_idx = callee_func.func_id
                            self.func_adj_matrix[caller_idx][callee_idx] = 1
                            print(f"Function {caller_func.func_name} calls {callee_func.func_name} in {caller_file.file_name}")
            except SyntaxError:
                continue

    def _find_file_by_path(self, file_path: str) -> Optional[File]:
        """通过文件路径查找文件对象"""
        for file in self.files:
            if file.file_path == file_path:
                return file
        return None

    def _build_symbol_table(self, file: File) -> Dict[str, object]:
        """构建当前文件的符号表"""
        symbol_table = {}
        
        # 添加当前文件中的函数
        for func in file.func_list:
            symbol_table[func.func_name] = func
        
        # 添加导入的函数和模块
        for entry in file.import_list:
            # 处理from...import
            if entry.imported_names:
                target_module = entry.module
                if target_module in self.module_to_file_map:
                    target_file = self.module_to_file_map[target_module]
                    for name, asname in entry.imported_names:
                        # 在目标文件中查找函数
                        for func in target_file.func_list:
                            if func.func_name == name:
                                key = asname if asname else name
                                symbol_table[key] = func
                                break
            
            # 处理标准import
            else:
                module_name = entry.module
                if module_name in self.module_to_file_map:
                    key = entry.asname if entry.asname else module_name.split('.')[-1]
                    symbol_table[key] = self.module_to_file_map[module_name]
        
        return symbol_table

    def _resolve_function_call(self, node: ast.Call, symbol_table: Dict, file_func_map: Dict) -> Optional[Function]:
        """解析函数调用节点"""
        # 处理直接函数名调用 (e.g., func())
        if isinstance(node.func, ast.Name):
            func_name = node.func.id
            if func_name in symbol_table:
                obj = symbol_table[func_name]
                if isinstance(obj, Function):
                    return obj
        
        # 处理属性调用 (e.g., module.func())
        elif isinstance(node.func, ast.Attribute):
            # 解析模块部分
            if isinstance(node.func.value, ast.Name):
                module_name = node.func.value.id
                func_name = node.func.attr
                
                if module_name in symbol_table:
                    obj = symbol_table[module_name]
                    
                    # 如果是导入的模块
                    if isinstance(obj, File):
                        # 在模块文件中查找函数
                        if obj.file_path in file_func_map:
                            func_map = file_func_map[obj.file_path]
                            if func_name in func_map:
                                return func_map[func_name]
        
        return None

In [194]:
# E:\Projects\codemap-master\SA_Python\data\MetaGPT-main\metagpt
base_dir = "E:/Projects/codemap-master/SA_Python/data/MetaGPT-main/metagpt"
analyzer = CodeAnalyzer(base_dir)
files, functions, file_adj_matrix, func_adj_matrix = analyzer.analyze_project()
print(f"Found {len(files)} files and {len(functions)} functions")

From import: from pathlib import [('Path', 'Path')]
From import: from typing import [('Dict', 'Dict'), ('Iterable', 'Iterable'), ('List', 'List'), ('Literal', 'Literal'), ('Optional', 'Optional')]
From import: from pydantic import [('BaseModel', 'BaseModel'), ('Field', 'Field'), ('model_validator', 'model_validator')]
From import: from metagpt.configs.browser_config import [('BrowserConfig', 'BrowserConfig')]
From import: from metagpt.configs.embedding_config import [('EmbeddingConfig', 'EmbeddingConfig')]
From import: from metagpt.configs.exp_pool_config import [('ExperiencePoolConfig', 'ExperiencePoolConfig')]
From import: from metagpt.configs.llm_config import [('LLMConfig', 'LLMConfig'), ('LLMType', 'LLMType')]
From import: from metagpt.configs.mermaid_config import [('MermaidConfig', 'MermaidConfig')]
From import: from metagpt.configs.omniparse_config import [('OmniParseConfig', 'OmniParseConfig')]
From import: from metagpt.configs.redis_config import [('RedisConfig', 'RedisConfig

In [195]:
# 将Function对象中的func_file字符串去E:/Projects/codemap-master/SA_Python/data/
def clean_function_file_paths(functions: List[Function], base_path: str):
    
    """将Function对象中的func_file转为相对路径的文件名"""
    for func in functions:
        print("before:", func.func_file)
        func.func_file =  str(func.func_file).replace(base_path, "")
        print("after:", func.func_file)
base_path = "E:/Projects/codemap-master/SA_Python/data/MetaGPT-main/"
clean_function_file_paths(functions, base_path)

# 打印前五个函数对象
for func in functions[:5]:
    print(func.func_id)
    print(f"Function: {func.func_name}, Full Name: {func.func_fullName}, File: {func.func_file}")
    print(f"  Description: {func.func_desc}")
    print(f"  Code: {func.func_code}")

before: E:/Projects/codemap-master/SA_Python/data/MetaGPT-main/metagpt\config2.py
after: metagpt\config2.py
before: E:/Projects/codemap-master/SA_Python/data/MetaGPT-main/metagpt\config2.py
after: metagpt\config2.py
before: E:/Projects/codemap-master/SA_Python/data/MetaGPT-main/metagpt\config2.py
after: metagpt\config2.py
before: E:/Projects/codemap-master/SA_Python/data/MetaGPT-main/metagpt\config2.py
after: metagpt\config2.py
before: E:/Projects/codemap-master/SA_Python/data/MetaGPT-main/metagpt\config2.py
after: metagpt\config2.py
before: E:/Projects/codemap-master/SA_Python/data/MetaGPT-main/metagpt\config2.py
after: metagpt\config2.py
before: E:/Projects/codemap-master/SA_Python/data/MetaGPT-main/metagpt\config2.py
after: metagpt\config2.py
before: E:/Projects/codemap-master/SA_Python/data/MetaGPT-main/metagpt\config2.py
after: metagpt\config2.py
before: E:/Projects/codemap-master/SA_Python/data/MetaGPT-main/metagpt\config2.py
after: metagpt\config2.py
before: E:/Projects/codemap-

In [196]:
# 将Function对象中的func_fullName改为类似MetaGPT-main.setup.run的结构
import os
for func in functions:
    full_name = func.func_file.replace('/', '.').replace(os.sep, '.')
    if full_name.endswith('.py'):
        full_name = full_name[:-3]
    func.func_fullName = f"{full_name}.{func.func_name}"
    print(f"Function: {func.func_name}, Full Name: {func.func_fullName}")

Function: merge_dict, Full Name: metagpt.config2.merge_dict
Function: check_project_path, Full Name: metagpt.config2.check_project_path
Function: from_home, Full Name: metagpt.config2.from_home
Function: default, Full Name: metagpt.config2.default
Function: from_llm_config, Full Name: metagpt.config2.from_llm_config
Function: update_via_cli, Full Name: metagpt.config2.update_via_cli
Function: extra, Full Name: metagpt.config2.extra
Function: extra, Full Name: metagpt.config2.extra
Function: get_openai_llm, Full Name: metagpt.config2.get_openai_llm
Function: get_azure_llm, Full Name: metagpt.config2.get_azure_llm
Function: get_metagpt_package_root, Full Name: metagpt.const.get_metagpt_package_root
Function: get_metagpt_root, Full Name: metagpt.const.get_metagpt_root
Function: __init__, Full Name: metagpt.context.__init__
Function: __getattr__, Full Name: metagpt.context.__getattr__
Function: __setattr__, Full Name: metagpt.context.__setattr__
Function: __delattr__, Full Name: metagpt.co

In [206]:
# 查看file_adj_matrix和func_adj_matrix有多少非零值
print(f"File adjacency matrix non-zero values: {np.count_nonzero(file_adj_matrix)}")
print(f"Function adjacency matrix non-zero values: {np.count_nonzero(func_adj_matrix)}")

File adjacency matrix non-zero values: 1469
Function adjacency matrix non-zero values: 196


In [204]:
# 将邻接矩阵转换为 持久化存储
func_adj_matrix_df = pd.DataFrame(func_adj_matrix)
func_adj_matrix_df.to_csv('func_adj_matrix.csv', index=False, header=False)

In [205]:
# 将邻接矩阵转换为 持久化存储
file_adj_matrix_df = pd.DataFrame(func_adj_matrix)
file_adj_matrix_df.to_csv('file_adj_matrix.csv', index=False, header=False)

In [207]:
# 打印前五个文件对象
for file in files[:5]:
    print(f"File ID: {file.file_id}, Name: {file.file_name}, Path: {file.file_path}")
    print(f"  Description: {file.file_desc}")
    print(f"  Code: {file.file_code[:100]}...")  # 打印前100个字符
    print(f"  Number of functions: {len(file.func_list)}")
    print(f"  Imports: {[entry.module for entry in file.import_list]}")

File ID: 0, Name: config2.py, Path: E:/Projects/codemap-master/SA_Python/data/MetaGPT-main/metagpt\config2.py
  Description: 
@Time    : 2024/1/4 01:25
@Author  : alexanderwu
@File    : config2.py

  Code: #!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
@Time    : 2024/1/4 01:25
@Author  : alexanderwu
@...
  Number of functions: 10
  Imports: []
File ID: 1, Name: const.py, Path: E:/Projects/codemap-master/SA_Python/data/MetaGPT-main/metagpt\const.py
  Description: 
  Code: #!/usr/bin/env python
# -*- coding: utf-8 -*-

import os
from pathlib import Path

from loguru impor...
  Number of functions: 2
  Imports: []
File ID: 2, Name: context.py, Path: E:/Projects/codemap-master/SA_Python/data/MetaGPT-main/metagpt\context.py
  Description: 
@Time    : 2024/1/4 16:32
@Author  : alexanderwu
@File    : context.py

  Code: #!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
@Time    : 2024/1/4 16:32
@Author  : alexanderwu
@...
  Number of functions: 13
  Imports: []
File ID: 3, Name: context_m

In [208]:
base_path = "E:/Projects/codemap-master/SA_Python/data/MetaGPT-main/"
"""将File对象中的file_path转为相对路径的文件名"""
for file in files:
    print("before:", file.file_path)
    file.file_path =  str(file.file_path).replace(base_path, "")
    print("after:", file.file_path)

before: E:/Projects/codemap-master/SA_Python/data/MetaGPT-main/metagpt\config2.py
after: MetaGPT-main/metagpt\config2.py
before: E:/Projects/codemap-master/SA_Python/data/MetaGPT-main/metagpt\const.py
after: MetaGPT-main/metagpt\const.py
before: E:/Projects/codemap-master/SA_Python/data/MetaGPT-main/metagpt\context.py
after: MetaGPT-main/metagpt\context.py
before: E:/Projects/codemap-master/SA_Python/data/MetaGPT-main/metagpt\context_mixin.py
after: MetaGPT-main/metagpt\context_mixin.py
before: E:/Projects/codemap-master/SA_Python/data/MetaGPT-main/metagpt\document.py
after: MetaGPT-main/metagpt\document.py
before: E:/Projects/codemap-master/SA_Python/data/MetaGPT-main/metagpt\llm.py
after: MetaGPT-main/metagpt\llm.py
before: E:/Projects/codemap-master/SA_Python/data/MetaGPT-main/metagpt\logs.py
after: MetaGPT-main/metagpt\logs.py
before: E:/Projects/codemap-master/SA_Python/data/MetaGPT-main/metagpt\repo_parser.py
after: MetaGPT-main/metagpt\repo_parser.py
before: E:/Projects/codemap-

In [209]:
# 查看0号文件的import的文件
for i in range(len(files)):
    if(file_adj_matrix[0][i] == 1):
        print(f"File {files[0].file_name} imports {files[i].file_name}")

File config2.py imports const.py
File config2.py imports browser_config.py
File config2.py imports embedding_config.py
File config2.py imports exp_pool_config.py
File config2.py imports llm_config.py
File config2.py imports mermaid_config.py
File config2.py imports omniparse_config.py
File config2.py imports redis_config.py
File config2.py imports role_custom_config.py
File config2.py imports role_zero_config.py
File config2.py imports s3_config.py
File config2.py imports search_config.py
File config2.py imports workspace_config.py
File config2.py imports yaml_model.py


In [219]:
# 生成总结函数的提示词

Func_summary_template = """\nYou are a software engineer who is reverse engineering the code in a system to extract its design requirements and functional descriptions. 
code is below:
{code}
A. Project context positioning
- Module Attribution analysis:
{folder_structure}
- Dependency graph:
{dependence}
B. Function functionality destructuring
- Input/output parameter analysis: List and explain the data structure and purpose of all inputs and outputs
- Core logic flowchart: flowchart describing key processing steps in natural language
- Exception handling mechanism: error handling logic and boundary conditions are identified and illustrated
- If you can figure out the action object of the function, specify it; if you can't, leave it out
# Task:
-Give a description of the function based on the AB step, and extrapolate the functional requirements from the code implementation
-Non-functional requirements identification: infer non-functional requirements such as performance and security as reflected by code constraints
-Answer exactly as the code says. Don't introduce extra information
-Use the following format to answer:
Please return the response in the following JSON format:
{{
    "func_desc": "Function description",
    "func_flow": "Function flowchart",
    "func_notf": "Non-functional requirements",
}}
"""

In [220]:
class func_response(BaseModel):
    func_desc: str
    func_flow: str
    func_notf: str
    class Config:
        extra = "forbid"  # 严格禁止额外字段

In [226]:
# 生成函数描述
from openai import OpenAI
import time
import json

client = OpenAI(
    # 修改api_key为网站生成的令牌
    api_key="sk-PrTxn5LU0UxWYip660910e6196944e28B57fCe19DeD9E6C0",
    base_url="https://api.openai.com/v1"
)
client.base_url="https://ai-yyds.com/v1"

for function in functions[1062:1063]:
    depencence = ""
    for j in range(1, len(functions)):
        if func_adj_matrix[function.func_id][j] != 0:
            depencence += functions[j].func_fullName + "\n" + functions[j].func_code + "\n"
    prompt = Func_summary_template.format(
        code=function.func_code,
        folder_structure=function.func_fullName,
        dependence=depencence
    )
    try:
        # 返回的结果是一个json数据，包含 func_desc、func_flow 和 func_notf 字段
        # 解析json数据为字典对象
        response = client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model="gpt-4o-mini-2024-07-18",   #deepseek-r1
            response_format={"type": "json_object"},  # 强制要求返回JSON格式
            temperature=0.3, # 降低随机性
            top_p=0.95, # 保持一定的创造性
            max_tokens=800, # 预留充足响应空间
            frequency_penalty=0.5, # 抑制重复内容
            presence_penalty=0.2, # 鼓励关键术语出现 
            timeout=60  # 设置超时时间为60秒
        )
        # 解析响应内容
        # 确保响应内容是 JSON 格式
        # 解析响应内容
        json_str = response.choices[0].message.content
        # 移除可能干扰JSON解析的代码块标记
        json_str = json_str.replace("```json", "").replace("```", "")
        result_dict = json.loads(json_str)
        print(result_dict)
        allowed_keys = {"func_desc", "func_flow", "func_notf"}
        result_dict = {k: v for k, v in result_dict.items() if k in allowed_keys}
        if "func_notf" not in result_dict:
            result_dict["func_notf"] = ""
        elif isinstance(result_dict.get("func_notf"), dict):
            result_dict["func_notf"] = json.dumps(result_dict["func_notf"], ensure_ascii=False)
        result = func_response.model_validate(result_dict)  # 改用 model_validate
        function.func_desc = result.func_desc
        function.func_flow = result.func_flow
        function.func_notf = result.func_notf
    except json.JSONDecodeError as e:
        print(f"JSON解析失败: {e}")
    except ValidationError as e:
        print(f"Pydantic验证失败: {e}")
    except Exception as e:
        print(f"其他错误: {e}")
    # 打印函数描述
    print(f"Function ID: {function.func_id}, Name: {function.func_name}")
    print(f"Description: {function.func_desc}")
    print(f"Flow: {function.func_flow}")
    print(f"Non-functional requirements: {function.func_notf}")
    time.sleep(0.2)

{'func_desc': "The function 'make_step' takes a 2D list 'm' and an integer 'k'. It searches for occurrences of 'k' in the matrix 'm'. For each occurrence, it checks the four adjacent cells (up, left, down, right) to see if they are equal to 0 in both 'm' and another matrix 'a'. If an adjacent cell meets these conditions, it sets that cell in 'm' to 'k + 1'.", 'func_flow': "1. Iterate over each row in matrix m. 2. For each element in the row, check if it equals k. 3. If found, check the cell above; if it's within bounds and both m and a have a value of 0, set that cell to k + 1. 4. Check the cell to the left; if it's within bounds and both m and a have a value of 0, set that cell to k + 1. 5. Check the cell below; if it's within bounds and both m and a have a value of 0, set that cell to k + 1. 6. Check the cell to the right; if it's within bounds and both m and a have a value of 0, set that cell to k + 1.", 'func_notf': 'The function should efficiently handle large matrices without exc

In [227]:
# 将Function对象转换为csv文件
def functions_to_csv(functions, filename):
    data = []
    for function in functions:
        data.append({
            "func_id": function.func_id,
            "func_name": function.func_name,
            "func_desc": function.func_desc,
            "func_file": function.func_file,
            "func_fullName": function.func_fullName,
            "func_flow": function.func_flow,
            "func_notf": function.func_notf,
        })
    df = pd.DataFrame(data)
    df.to_csv(filename, index=False, escapechar='\\', quoting=csv.QUOTE_ALL)  # 修改这一行

out_put_path = "chat-4o_functions.csv"
functions_to_csv(functions, out_put_path)

In [228]:
File_summary_template = """\nYou're an expert in Java architecture analysis, mapping code structure to functional requirements
file name is {file_name}
file path is {file_path}
code is below:
{code}
# Task:
- Answer exactly as the code says. Don't introduce extra information
- Please return the response in the following JSON format:
{{
    "file_desc": "Functional Description"
}}
example:
{one_shot}
"""

In [229]:
one_shot = """
code:
package edu.nd.dronology.services.core.remote;

import java.rmi.RemoteException;
import java.util.Collection;
import java.util.List;

import edu.nd.dronology.core.util.Waypoint;
import edu.nd.dronology.services.core.info.FlightInfo;
import edu.nd.dronology.services.core.info.FlightPlanInfo;
import edu.nd.dronology.services.core.util.DronologyServiceException;

/**
 * 
 * @author Michael Vierhauser
 * 
 */
public interface IFlightManagerRemoteService extends IRemoteableService {{

	void planFlight(String planName, List<Waypoint> wayPoints) throws RemoteException, Exception;

	void planFlight(String uavid, String planName, List<Waypoint> wayPoints) throws RemoteException, Exception;

	void returnToHome(String uavid) throws RemoteException, Exception;

	void takeoff(String uavid, double altitude) throws RemoteException, DronologyServiceException;

	void pauseFlight(String iavid) throws RemoteException, Exception;

	FlightInfo getFlightInfo(String uavId) throws RemoteException, Exception;

	Collection<FlightPlanInfo> getCurrentFlights() throws RemoteException;

	void cancelPendingFlights(String uavid) throws RemoteException, Exception;

}}

response:
{{
    "file_desc": "The IFlightManagerRemoteService interface enables remote control of drones over a network. It provides methods to plan drone flight paths by specifying waypoint lists. Unique drone IDs can be used to issue takeoff, pause, and return home commands. The interface also allows fetching real-time flight status and information for individual drones as well as an overview of all ongoing flights. By implementing this interface, a flight management system can remotely monitor and direct drones to carry out missions and data collection flights. The separation of flight control logic from the drones provides flexibility to integrate drones from various vendors into the system."
}}
"""

class file_response(BaseModel):
    file_desc: str
    class Config:
        extra = "forbid"  # 严格禁止额外字段

In [230]:
# 打印每个文件中的函数列表
for file in files:
    print(f"File: {file.file_name}, Path: {file.file_path}")
    for func in file.func_list:
        print(f"  Function: {func.func_name}, Full Name: {func.func_fullName}")

File: config2.py, Path: MetaGPT-main/metagpt\config2.py
  Function: merge_dict, Full Name: metagpt.config2.merge_dict
  Function: check_project_path, Full Name: metagpt.config2.check_project_path
  Function: from_home, Full Name: metagpt.config2.from_home
  Function: default, Full Name: metagpt.config2.default
  Function: from_llm_config, Full Name: metagpt.config2.from_llm_config
  Function: update_via_cli, Full Name: metagpt.config2.update_via_cli
  Function: extra, Full Name: metagpt.config2.extra
  Function: extra, Full Name: metagpt.config2.extra
  Function: get_openai_llm, Full Name: metagpt.config2.get_openai_llm
  Function: get_azure_llm, Full Name: metagpt.config2.get_azure_llm
File: const.py, Path: MetaGPT-main/metagpt\const.py
  Function: get_metagpt_package_root, Full Name: metagpt.const.get_metagpt_package_root
  Function: get_metagpt_root, Full Name: metagpt.const.get_metagpt_root
File: context.py, Path: MetaGPT-main/metagpt\context.py
  Function: __init__, Full Name: met

In [231]:
# 替换file_code中函数代码为函数描述
for file in files:
    for function in file.func_list:
        file.file_discode = file.file_code.replace(function.func_code, "/** method name:"+function.func_name+"\n * method description:"+function.func_desc+"\n * method flow:"+function.func_flow+"\n*/")

# 打印文件代码
for file in files[:5]:
    print(f"File ID: {file.file_id}, Name: {file.file_name}, Path: {file.file_path}, Description: {file.file_desc}")
    print(f"Code Snippet:\n{file.file_discode}\n")

File ID: 0, Name: config2.py, Path: MetaGPT-main/metagpt\config2.py, Description: 
@Time    : 2024/1/4 01:25
@Author  : alexanderwu
@File    : config2.py

Code Snippet:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
@Time    : 2024/1/4 01:25
@Author  : alexanderwu
@File    : config2.py
"""
import os
from pathlib import Path
from typing import Dict, Iterable, List, Literal, Optional

from pydantic import BaseModel, Field, model_validator

from metagpt.configs.browser_config import BrowserConfig
from metagpt.configs.embedding_config import EmbeddingConfig
from metagpt.configs.exp_pool_config import ExperiencePoolConfig
from metagpt.configs.llm_config import LLMConfig, LLMType
from metagpt.configs.mermaid_config import MermaidConfig
from metagpt.configs.omniparse_config import OmniParseConfig
from metagpt.configs.redis_config import RedisConfig
from metagpt.configs.role_custom_config import RoleCustomConfig
from metagpt.configs.role_zero_config import RoleZeroConfig
from metagpt.config

In [232]:
client = OpenAI(
    # 修改api_key为网站生成的令牌
    api_key="sk-PrTxn5LU0UxWYip660910e6196944e28B57fCe19DeD9E6C0",
    base_url="https://api.openai.com/v1"
)
client.base_url="https://ai-yyds.com/v1"


# 生成文件描述
for file in files:
    prompt = File_summary_template.format(
        file_name=file.file_name,
        file_path=file.file_path,
        code=file.file_discode, # 将代码中的函数替换为函数描述
        one_shot=one_shot
    )
    try:
        # 返回的结果是一个json数据，包含 file_des 字段
        response = client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model="gpt-4o-mini-2024-07-18",
            response_format={"type": "json_object"},  # 强制要求返回JSON格式
            temperature=0.3, # 降低随机性
            top_p=0.95, # 保持一定的创造性
            max_tokens=800, # 预留充足响应空间
            frequency_penalty=0.5, # 抑制重复内容
            presence_penalty=0.2 # 鼓励关键术语出现 
        )
        json_str = response.choices[0].message.content
        # 移除可能干扰JSON解析的代码块标记
        json_str = json_str.replace("```json", "").replace("```", "")
        result = file_response.model_validate(json.loads(json_str))  # 改用 model_validate
        file.file_desc = result.file_desc
    except json.JSONDecodeError as e:
        print(f"JSON解析失败: {e}")
    except ValidationError as e:
        print(f"Pydantic验证失败: {e}")
    except Exception as e:
        print(f"其他错误: {e}")
    # 打印函数描述
    print(f"File ID: {file.file_id}, Name: {file.file_name}, des: {file.file_desc}")

File ID: 0, Name: config2.py, des: The config2.py file defines configuration classes for the MetaGPT application using Pydantic models. It includes CLI parameters and various configurations such as LLM, embedding, omniparse, search, browser settings, storage options (S3 and Redis), and experience pool parameters. The Config class also provides methods to load configurations from files or environment variables, allowing for dynamic updates based on user input or defaults. Additionally, it includes functionality to manage roles and their custom configurations. Overall, this file serves as a centralized configuration management system for the MetaGPT application.
File ID: 1, Name: const.py, des: The const.py file defines various constants and paths used throughout the MetaGPT project. It includes functions to determine the project root directory based on environment variables and checks for specific files to confirm the project's structure. The file also sets up configurations for differe

In [233]:
# 将file类中存储到csv文件中
def save_files_to_csv(files, filename):
    data = []
    for file in files:
        data.append({
            "file_id": file.file_id,
            "file_name": file.file_name,
            "file_path": file.file_path,
            "file_desc": file.file_desc
        })
    df = pd.DataFrame(data)
    df.to_csv(filename, index=False)
# 保存到 CSV 文件
save_files_to_csv(files, "chat-4o_CoT1s_java_files.csv")

In [234]:
from sentence_transformers import SentenceTransformer
from dataclasses import dataclass
from typing import List
import numpy as np
import torch

In [235]:
class file_Cluster:
    cluster_id: int
    cluster_desc: str
    cluster_file_list: List[File]

    def __init__(self, cluster_id, cluster_desc, cluster_file_list):
        self.cluster_id = cluster_id
        self.cluster_desc = cluster_desc
        self.cluster_file_list = cluster_file_list

In [236]:
# 计算file之间的文本相似度
model = SentenceTransformer('all-mpnet-base-v2')

for file in files:
    # 计算文件的文本向量
    file.file_txt_vector = model.encode(file.file_desc).tolist()

In [237]:
# 打印函数的文本向量
for file in files[:5]:
    print(f"File ID: {file.file_id}, Name: {file.file_name}, Text Vector: {file.file_txt_vector}")

File ID: 0, Name: config2.py, Text Vector: [0.01771806925535202, -0.05693667754530907, -0.0005861014360561967, -0.005675699096173048, 0.020210886374115944, 0.022148465737700462, 0.04406179115176201, -0.00524028018116951, -0.06695821136236191, -0.030705517157912254, -0.020242655649781227, 0.028111396357417107, -0.021532490849494934, 0.034688133746385574, 0.006765484809875488, -0.011553838849067688, 0.005396612454205751, -0.022686555981636047, -0.04163295775651932, 0.02824692614376545, -0.006505632773041725, 0.022257938981056213, 0.01680184341967106, 0.02704724669456482, 0.021974975243210793, -0.0030491536017507315, 0.006406907923519611, 0.03662492334842682, 0.019618669524788857, -0.05991153046488762, -0.007582052610814571, -0.022111931815743446, -0.0020067915320396423, 0.016943801194429398, 2.1831078811374027e-06, -0.007210856303572655, -0.028868604451417923, -0.0016880377661436796, -0.01536681316792965, 0.023361962288618088, 0.04569116234779358, 0.022777345031499863, 0.0247560590505599

In [238]:
# 得到文本相似度矩阵
def compute_similarity_matrix(files):
    # 提取文件向量
    txt_vectors = [file.file_txt_vector for file in files]
    normalized_vectors = txt_vectors / np.linalg.norm(txt_vectors, axis=1, keepdims=True)
    # 计算相似度矩阵
    similarity_matrix = np.dot(normalized_vectors, np.transpose(normalized_vectors))
    # 将相似度矩阵的值设置为0-1之间
    similarity_matrix = (similarity_matrix + 1) / 2
    return similarity_matrix

In [239]:
similarity_matrix = compute_similarity_matrix(files)
link_matrix = file_adj_matrix.copy()
# 将link_matrix变成对称矩阵
for i in range(len(link_matrix)):
    for j in range(i + 1, len(link_matrix)):
        if link_matrix[i][j] != 0 or link_matrix[j][i] != 0:
            link_matrix[i][j] = 1
            link_matrix[j][i] = 1

# 统计link_matrix的非零值数量
non_zero_count = np.count_nonzero(link_matrix)
print(f"Number of non-zero values in the link matrix: {non_zero_count}")
# 计算最终的权重矩阵
weight_matrix = similarity_matrix*0.25 + link_matrix*0.75

Number of non-zero values in the link matrix: 2910


In [240]:
# 安装更先进的Leiden算法
# pip install leidenalg

import leidenalg
from igraph import Graph

In [241]:
def leiden_clustering(weight_matrix):
    # 去除对角线元素
    np.fill_diagonal(weight_matrix, 0)
    G = Graph.Weighted_Adjacency(weight_matrix.tolist(), mode="UNDIRECTED")
    
    partition = leidenalg.find_partition(
        G, 
        partition_type=leidenalg.CPMVertexPartition,  # 分辨率参数敏感的分区类型
        resolution_parameter=0.17,   
        weights="weight",
        n_iterations=-1  # 无限迭代直到收敛
    )
    # 计算模块度
    Q = partition.quality()
    print(f"Modularity: {Q}")
    return partition.membership
# 将划分结果转换为file_Cluster对象
def convert_to_file_clusters(partition, files):
    clusters = {}
    for file_id, cluster_id in enumerate(partition):
        if cluster_id not in clusters:
            clusters[cluster_id] = file_Cluster(cluster_id, "", [])
        # file_id从1开始，但是files的索引从0开始 所以不用处理
        clusters[cluster_id].cluster_file_list.append(files[file_id])
    
    return clusters.values()

# 使用Leiden算法进行聚类
partition = leiden_clustering(weight_matrix)
clusters = convert_to_file_clusters(partition, files)


Modularity: 1792.199865812767


In [242]:
# 打印每个聚类的文件列表
for cluster in clusters:
    print(f"Cluster ID: {cluster.cluster_id}, Files: {[file.file_name for file in cluster.cluster_file_list]}")

Cluster ID: 0, Files: ['config2.py', 'const.py', 'context.py', 'context_mixin.py', 'logs.py', 'repo_parser.py', 'schema.py', 'software_company.py', 'team.py', 'action.py', 'action_node.py', 'action_output.py', 'add_requirement.py', 'analyze_requirements.py', 'debug_error.py', 'design_api.py', 'design_api_an.py', 'execute_task.py', 'extract_readme.py', 'fix_bug.py', 'generate_questions.py', 'import_repo.py', 'invoice_ocr.py', 'prepare_documents.py', 'prepare_interview.py', 'project_management.py', 'project_management_an.py', 'rebuild_class_view.py', 'rebuild_sequence_view.py', 'research.py', 'run_code.py', 'search_and_summarize.py', 'search_enhanced_qa.py', 'skill_action.py', 'summarize_code.py', 'talk_action.py', 'write_code.py', 'write_code_an_draft.py', 'write_code_plan_and_change_an.py', 'write_code_review.py', 'write_docstring.py', 'write_prd.py', 'write_prd_an.py', 'write_prd_review.py', 'write_review.py', 'write_teaching_plan.py', 'write_test.py', 'write_tutorial.py', '__init__.p

In [243]:
class method_Cluster:
    cluster_id: int
    cluster_desc: str
    cluster_func_list: List[Function]

    def __init__(self, cluster_id, cluster_desc, cluster_func_list):
        self.cluster_id = cluster_id
        self.cluster_desc = cluster_desc
        self.cluster_func_list = cluster_func_list

In [244]:
class Feature:
    cluster_id: int
    feature_id: int
    feature_desc: str
    feature_flow: str = ""
    feature_notf: str = ""
    feature_func_list: List[Function]

    def __init__(self, cluster_id,feature_id, feature_desc, feature_func_list):
        self.cluster_id = cluster_id
        self.feature_id = feature_id
        self.feature_desc = feature_desc
        self.feature_func_list = feature_func_list

In [245]:
method_clusters = []
# 将clusters展开到函数层
for cluster in clusters:
    list = []
    for file in cluster.cluster_file_list:
        for function in file.func_list:
            # 将函数添加到聚类中
            function.func_txt_vector = model.encode(function.func_desc).tolist()
            list.append(function)
    method_cluster = method_Cluster(cluster.cluster_id, "", list)
    method_clusters.append(method_cluster)

# 打印每个聚类的函数列表
for method_cluster in method_clusters:
    print(f"Cluster ID: {method_cluster.cluster_id}, Functions: {[f.func_name for f in method_cluster.cluster_func_list]}")

Cluster ID: 0, Functions: ['merge_dict', 'check_project_path', 'from_home', 'default', 'from_llm_config', 'update_via_cli', 'extra', 'extra', 'get_openai_llm', 'get_azure_llm', 'get_metagpt_package_root', 'get_metagpt_root', '__init__', '__getattr__', '__setattr__', '__delattr__', 'set', 'get', 'remove', 'new_environ', '_select_costmanager', 'llm', 'llm_with_cost_manager_from_llm_config', 'serialize', 'deserialize', 'validate_context_mixin_extra', '_process_context_mixin_extra', 'set', 'set_context', 'set_config', 'set_llm', 'config', 'config', 'context', 'context', 'llm', 'llm', 'define_log_level', 'log_llm_stream', 'log_tool_output', 'set_llm_stream_logfunc', 'set_tool_output_logfunc', 'set_human_input_func', 'create_llm_stream_queue', 'get_llm_stream_queue', '_llm_stream_log', 'is_func', 'parse', 'parse_compositions', '_split_literal', 'sort', 'sort', 'parse', 'sort', 'parse', '_parse_name', '_parse_args', '_parse_file', 'extract_class_and_function_info', 'generate_symbols', 'genera

In [246]:
# 得到文本相似度矩阵
def compute_similarity_matrix(method_cluster):
    # 提取文件向量
    txt_vectors = [func.func_txt_vector for func in method_cluster.cluster_func_list]
    # 归一化
    txt_vectors = np.array(txt_vectors)
    normalized_vectors = txt_vectors / np.linalg.norm(txt_vectors, axis=1, keepdims=True)
    # 计算相似度矩阵
    similarity_matrix = np.dot(normalized_vectors, np.transpose(normalized_vectors))
    # 将相似度矩阵的值设置为0-1之间
    similarity_matrix = (similarity_matrix + 1) / 2
    return similarity_matrix

In [247]:
def compute_link(method_cluster):
    link_matrix = np.zeros((len(method_cluster.cluster_func_list), len(method_cluster.cluster_func_list)))
    i=0 
    for function in method_cluster.cluster_func_list:
        j=0
        for other_function in method_cluster.cluster_func_list:
            if i == j:
                j += 1
                continue
            if func_adj_matrix[function.func_id][other_function.func_id] != 0:
                # 如果存在call和caller关系，设置邻接矩阵的对应位置为 1
                link_matrix[i][j] = 1
                link_matrix[j][i] = 1
            j += 1
        i += 1
    return link_matrix

In [248]:
def fuc_leiden_clustering(weight_matrix, resolution_parameter):
    # 去除对角线元素
    np.fill_diagonal(weight_matrix, 0)
    G = Graph.Weighted_Adjacency(weight_matrix.tolist(), mode="UNDIRECTED")
    
    partition = leidenalg.find_partition(
        G, 
        partition_type=leidenalg.CPMVertexPartition,  # 分辨率参数敏感的分区类型
        resolution_parameter= resolution_parameter,
        weights="weight",
        n_iterations=-1  # 无限迭代直到收敛
    )
    # 计算模块度
    Q = partition.quality()
    print(f"Modularity: {Q}")
    return partition.membership

In [249]:
# 将函数进行聚类
def cluster_functions(method_cluster, weight_parameter, resolution_parameter):
    # 计算函数之间的文本相似度矩阵
    similarity_matrix = compute_similarity_matrix(method_cluster)
    # 计算链接矩阵
    link_matrix = compute_link(method_cluster)
    # 计算最终的权重矩阵
    weight_matrix = similarity_matrix * weight_parameter + link_matrix * (1 - weight_parameter)
    # 使用leiden算法进行聚类
    partition = fuc_leiden_clustering(weight_matrix, resolution_parameter)
    
    # 将划分结果转换为Feature对象
    clusters = {}
    for func_id, cluster_id in enumerate(partition):
        if cluster_id not in clusters:
            clusters[cluster_id] = Feature(method_cluster.cluster_id, 0,"", [])
        clusters[cluster_id].feature_func_list.append(method_cluster.cluster_func_list[func_id])
    
    return clusters.values()

In [252]:
feature_list = []
for method_cluster in method_clusters:
    if len(method_cluster.cluster_func_list) ==0:
        continue
    elif len(method_cluster.cluster_func_list) == 1:
        # 如果只有一个函数，则直接创建一个feature_list
        feature = Feature(method_cluster.cluster_id, 30, "", [method_cluster.cluster_func_list[0]])
        feature_list.append(feature)
    else:
        sub_feature_list = cluster_functions(method_cluster, 0.25, 0.17)
        feature_list.extend(sub_feature_list)

# 为feature_list中的每个feature分配唯一的ID
for i, feature in enumerate(feature_list):
    feature.feature_id = i + 1  # 从 1 开始编号

# 打印每个聚类的函数列表
for feature in feature_list:
    print(f"Feature ID: {feature.feature_id}, Functions: {[f.func_name for f in feature.feature_func_list]}")
        

Modularity: 692.4137303909348
Modularity: 44.077762102769
Modularity: 177.0048320665315
Modularity: 0.9709597134094071
Modularity: 0.11003277431118086
Modularity: 3.982072461648526
Modularity: 0.16777664238332557
Modularity: 45.004337060578884
Modularity: 12.859411828752876
Modularity: 0.06191717709955924
Modularity: 0.0730357065380553
Modularity: 4.579867470939127
Modularity: 0.0
Modularity: 8.956721892031295
Modularity: 214.24147713927363
Modularity: 9.262123428895336
Modularity: 6.070965444984034
Modularity: 0.25109823420958
Modularity: 10.273615843469667
Modularity: 4.242427332030933
Modularity: 3.578601888682122
Modularity: 1.2364202295835445
Modularity: 0.0
Modularity: 0.5556397958107633
Modularity: 0.5891560422753344
Feature ID: 1, Functions: ['merge_dict', 'default']
Feature ID: 2, Functions: ['check_project_path', 'from_home', 'extra', 'extra', '__init__', '__getattr__', '__setattr__', 'set', 'get', 'serialize', 'context', '__str__', 'check_instruct_content', 'ser_instruct_con

In [253]:
userstory_prompt = """
You are an engineer of a software system, and your goal is to reverse engineer design requirements from code. You will get a list of ids and code description in the system below.
# Code:
{code_content}
# Task:
Use the following steps to create Design requirements:
- Identify the actors of the use case, such as doctors, patients, webmasters, and so on;
- Group together related or overlapping operations in your code;
- Design requirements meet the following description:
* Design requirements are specific and detailed descriptions of features or functionality that a software development  project must include to achieve its goals.     It is an artifact that focuses on software design aspects,  such as the user interface, architecture,  and interactions between components.     This distinguishes it from other software artifacts such as functional  requirements, which focus on what the software should do, and non-functional requirements,  which focus on how the software should perform.
- Contains the basic elements of the usecase (roles/actions/values)
- Incorporate appropriate details from the code to ensure that the design requirements are clear and unambiguous.      Refer to the core Goals section to identify the necessary details to understand how the core user needs/goals are being  facilitated and/or the behaviors that are occurring.     All details must focus on the main objectives of the design  requirements.     Don't make up information.     All information must come from the provided code.

Please return the response in the following JSON format:
{{
"description": "Usecase description",
"flow": "Usecase events",
"notf":"Non-functional requirements (concerned with how the software should execute)"
}}

* For example:
{{
"description": "user inserts a new cultural object in the system",
"flow":   "1.     User activates new cultural good insertion.2.     System displays input form.     3.    User submits form  data.4.     System  validates data (invalid data triggers Errored use case).5.     User confirms operation.6.     System  persists cultural good",
"notf": "Rejects duplicate cultural heritage entries based on unique identifiers"
}}

"""

class usecase(BaseModel):
    description: str
    flow: str
    notf: str


In [254]:
# 生成函数描述
from openai import OpenAI
import time
import json

client = OpenAI(
    # 修改api_key为网站生成的令牌
    api_key="sk-PrTxn5LU0UxWYip660910e6196944e28B57fCe19DeD9E6C0",
    base_url="https://api.openai.com/v1"
)
client.base_url="https://ai-yyds.com/v1"

In [255]:
# 生成特征描述
for feature in feature_list:
    # 生成提示词
    code = ""
    # feature_func_list: List[Function]
    for function in feature.feature_func_list:
        code += "fuction name:"+str(function.func_fullName) + "\ndescription:" +str(function.func_desc) + "\nflow:"+str(function.func_flow)+"\nNon-functional requirements:" + str(function.func_notf)+  "\n"
    prompt = userstory_prompt.format(
        code_content=code
    )
    try:
        # 返回的结果是一个json数据，包含 description、flow 和 notf 字段
        response = client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model="gpt-4o-mini-2024-07-18",   #deepseek-r1
            response_format={"type": "json_object"},  # 强制要求返回JSON格式
            temperature=0.3, # 降低随机性
            top_p=0.95, # 保持一定的创造性
            max_tokens=800, # 预留充足响应空间
            frequency_penalty=0.5, # 抑制重复内容
            presence_penalty=0.2 # 鼓励关键术语出现 
        )
        json_str = response.choices[0].message.content
        print(json_str)
        # 移除可能干扰JSON解析的代码块标记
        json_str = json_str.replace("```json", "").replace("```", "")
        result = usecase.model_validate(json.loads(json_str))  # 改用 model_validate
        feature.feature_desc = result.description
        feature.feature_flow = result.flow
        feature.feature_notf = result.notf
    except json.JSONDecodeError as e:
        print(f"JSON解析失败: {e}")
    except ValidationError as e:
        print(f"Pydantic验证失败: {e}")
    except Exception as e:
        print(f"其他错误: {e}")
    # 打印函数描述
    print(f"Feature ID: {feature.feature_id}, Description: {feature.feature_desc}")

{
  "description": "The system merges multiple configuration dictionaries into a single cohesive configuration, prioritizing the latest values for duplicate keys and allowing for dynamic loading of default configurations from environment variables and YAML files.",
  "flow": "1. User requests to load the default configuration. 2. System checks if a reload is necessary based on the 'reload' parameter or if the configuration is already cached. 3. If reloading is required, system gathers configurations from environment variables and specified YAML file paths, along with any additional keyword arguments provided by the user. 4. System merges these configurations into a single dictionary using the merge_dict function, ensuring that later values overwrite earlier ones for duplicate keys. 5. The final merged configuration is cached to avoid future reloads unless explicitly requested again. 6. System returns the final merged configuration to the user.",
  "notf": "The system efficiently handle

In [259]:
# 将feature保存到csv文件中，其中一行数据只包括一个文件
def features_to_csv(features, filename):
    rows = []
    # 创建 DataFrame
    for feature in features:
        file_list = []
        for function in feature.feature_func_list:
        # 将函数的文件路径添加到列表中
            for file in files:
                if function.func_file == file.file_name:
                    # 判断文件路径是否已经存在于列表中
                    if file.file_name not in file_list:
                        file_list.append(file.file_name)
        for file_name in file_list:    
            rows.append ({ 
                "feature_id": feature.feature_id ,
                "cluster_id": feature.cluster_id ,
                "feature_desc": feature.feature_desc,
                "file_name": file_name,
                "feature_flow": feature.feature_flow,
                "feature_notf": feature.feature_notf,
            })
    df = pd.DataFrame(rows)

    # 保存为 CSV 文件
    df.to_csv(filename, index=False)
    print(f"Features saved to {filename}")

# 保存到 CSV 文件
features_to_csv(feature_list, "chat-4o_CoT1s_a=0.25_b=0.25_features_file.csv")

Features saved to chat-4o_CoT1s_a=0.25_b=0.25_features_file.csv


In [261]:
# 将feature保存到csv文件中，其中一行数据只包括一个函数


def features_to_csv(features, filename):
    rows = []
    # 创建 DataFrame
    for feature in features:
        for function in feature.feature_func_list:
        # 将函数的文件路径添加到列表中  
            rows.append ({ 
                "feature_id": feature.feature_id ,
                "cluster_id": feature.cluster_id ,
                "feature_desc": feature.feature_desc,
                "method_name": function.func_fullName,
                "feature_flow": feature.feature_flow,
                "feature_notf": feature.feature_notf,
            })
    df = pd.DataFrame(rows)

    # 保存为 CSV 文件
    df.to_csv(filename, index=False)
    print(f"Features saved to {filename}")

# 保存到 CSV 文件
features_to_csv(feature_list, "chat-4o_CoT1s_a=0.25_b=0.25_features_method.csv")

Features saved to chat-4o_CoT1s_a=0.25_b=0.25_features_method.csv


In [ ]:
merge_userstory_prompt = """
You are a professional software requirements analyst.Please follow these rules to analyze and merge requirements:
# sub-requirements list:
{sub_feature_list}
# Current system module list:
{module_list}
# TASK:
use the following steps to summary:
- Extract core operation objects (usually nouns) from each sub-requirement;
- Identify semantic relationships between objects (synonyms, hierarchical relations);
- For requirements involving different user roles (e.g., "Administrator deletes comment" vs "User adds comment"),  maintain the core module name while ensuring the implementation supports role differentiation.
- Merge requirements with same/core-related objects into one functional module;
- The generated module cannot already be present in the current system module list;

Please return the response in the following JSON format:
{{
"description": "module description"
}}

* For example:
Input:
1. Administrator deletes comment
2. Administrator reviews comment
3. Administrator queries backend comments
4. User gets first six comments
5. User gets replies by commentId
6. User gets comment
7. User adds comment
Output:
{{
"description": "Comment Module"
}}

"""
# Module
class module(BaseModel):
    description: str



In [281]:
# 将属于同method_cluster的feature合并
def merge_features_by_method_cluster(features, method_clusters):
    merged_features_des = []
    for method_cluster in method_clusters:
        # 获取属于同一method_cluster的feature
        related_features = [f for f in features if f.cluster_id == method_cluster.cluster_id]
        sub_feature_list = ""
        for i, feature in enumerate(related_features):
            sub_feature_list += f"{i+1}. {feature.feature_desc}\n"
        module_list = ""
        for i, merged_feature_des in enumerate(merged_features_des):
            module_list += f"{i+1}. {merged_feature_des}\n"
        prompt = merge_userstory_prompt.format(
            sub_feature_list=sub_feature_list,
            module_list=module_list
        )
        try:
            # 返回的结果是一个json数据，包含 description、flow 和 notf 字段
            response = client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model="gpt-4o-mini-2024-07-18",   #deepseek-r1
                response_format={"type": "json_object"},  # 强制要求返回JSON格式
                temperature=0.3, # 降低随机性
                top_p=0.95, # 保持一定的创造性
                max_tokens=800, # 预留充足响应空间
                frequency_penalty=0.5, # 抑制重复内容
                presence_penalty=0.2 # 鼓励关键术语出现 
            )
            json_str = response.choices[0].message.content
            print(json_str)
            # 移除可能干扰JSON解析的代码块标记
            json_str = json_str.replace("```json", "").replace("```", "")
            result_dict = json.loads(json_str)
            if "description" not in result_dict and isinstance(result_dict, dict):
                # 兼容 LLM 返回的 {模块名: 描述} 格式
                result_dict = {"description": next(iter(result_dict.values()))}
            result = module.model_validate(result_dict)
            merged_features_des.append(result.description)
            method_cluster.cluster_desc = result.description  # 更新method_cluster的描述
        except json.JSONDecodeError as e:
            print(f"JSON解析失败: {e}")
        except ValidationError as e:
            print(f"Pydantic验证失败: {e}")
        except Exception as e:
            print(f"其他错误: {e}")
        # 打印模块描述
        print(f"Module ID: {method_cluster.cluster_id}, Description: {method_cluster.cluster_desc}")
    

In [282]:
merge_features_by_method_cluster(feature_list, method_clusters)

{
  "description": "The system module provides comprehensive project management functionalities, allowing users to initialize and manage projects, configurations, and actions through a structured interface. It supports the merging of configuration dictionaries with prioritized values, dynamic loading from environment variables and YAML files, and efficient handling of project paths. Users can manage Git repositories for documentation, handle metadata, and execute tasks while maintaining dependencies. The module also includes capabilities for managing investments, team attributes, memory data serialization/deserialization, logging functionalities for monitoring actions and outputs, and user interactions with language models to generate responses based on configurations."
}
Module ID: 0, Description: The system module provides comprehensive project management functionalities, allowing users to initialize and manage projects, configurations, and actions through a structured interface. It 

In [283]:
# 将feature和module保存到csv文件中，其中一行数据只包括一个函数

def features_to_csv(features, filename):
    rows = []
    # 创建 DataFrame
    for feature in features:
        # 根据cluster_id获取对应的method_cluster
        method_cluster = next((mc for mc in method_clusters if mc.cluster_id == feature.cluster_id), None)
        for function in feature.feature_func_list:
        # 将函数的文件路径添加到列表中  
            rows.append ({ 
                "feature_id": feature.feature_id ,
                "cluster_id": feature.cluster_id ,
                "module_desc": method_cluster.cluster_desc,
                "feature_desc": feature.feature_desc,
                "method_name": function.func_fullName,
                "feature_flow": feature.feature_flow,
                "feature_notf": feature.feature_notf,
            })
    df = pd.DataFrame(rows)

    # 保存为 CSV 文件
    df.to_csv(filename, index=False)
    print(f"Features saved to {filename}")

# 保存到 CSV 文件
features_to_csv(feature_list, "chat-4o_CoT1s_a=0.25_b=0.25_detail_features_method.csv")

Features saved to chat-4o_CoT1s_a=0.25_b=0.25_detail_features_method.csv
